In [19]:
# python 3.13.9

In [1]:
import numpy as np 
import pandas as pd
import sys
import os

In [21]:
data='/Users/Tysia/Desktop/Edukacja/Informatyka/Magisterka/ml-1m'
rating_data=os.path.join(data,'ratings.dat')
movie_data=os.path.join(data,'movies.dat')
user_data=os.path.join(data,'users.dat')

ratings=pd.read_csv(
  rating_data,
  sep='::',
  engine='python',
  names=['user','item','rating','timestamp'],
  encoding='latin-1'   
)

movies=pd.read_csv(
    movie_data, 
    sep='::',
    engine='python',
    names=['item', 'title', 'genres'],
    encoding='latin-1'
)
users = pd.read_csv(
    user_data, 
    sep='::',
    engine='python',
    names=['user', 'gender', 'age', 'occupation', 'zipcode'],
    encoding='latin-1'
)

In [3]:
print(ratings.head())
print(movies.head())
print(users.head())

   user  item  rating  timestamp
0     1  1193       5  978300760
1     1   661       3  978302109
2     1   914       3  978301968
3     1  3408       4  978300275
4     1  2355       5  978824291
   item                               title                        genres
0     1                    Toy Story (1995)   Animation|Children's|Comedy
1     2                      Jumanji (1995)  Adventure|Children's|Fantasy
2     3             Grumpier Old Men (1995)                Comedy|Romance
3     4            Waiting to Exhale (1995)                  Comedy|Drama
4     5  Father of the Bride Part II (1995)                        Comedy
   user gender  age  occupation zipcode
0     1      F    1          10   48067
1     2      M   56          16   70072
2     3      M   25          15   55117
3     4      M   45           7   02460
4     5      M   25          20   55455


In [2]:
path = '/Users/Tysia/Desktop/Edukacja/Informatyka/Magisterka/Daisyrec/libraries/DaisyRec-v2.0-dev'

sys.path.append(path)

In [1]:
import torch
(torch.__version__)


'2.9.1+cpu'

In [23]:
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split  
#from daisy.utils.splitter import split_test
from daisy.model.KNNCFRecommender import ItemKNNCF, UserKNNCF
from daisy.model.PureSVDRecommender import PureSVD
from daisy.model.PopRecommender import MostPop
from daisy.model.MFRecommender import MF
from daisy.utils.metrics import Precision, Recall, NDCG
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tqdm import tqdm

In [ ]:
# ID muszą być liczbami od 0 do N-1, przemapowanie ID

df = ratings.copy()

# mapowanie user_id i item_id na ciągłe indeksy 0..N
encoderU = LabelEncoder()
encoderI = LabelEncoder()

df['user'] = encoderU.fit_transform(df['user'])
df['item'] = encoderI.fit_transform(df['item'])

num_users = len(encoderU.classes_)
num_items = len(encoderI.classes_)

In [25]:
import logging
# konfiguracja loggera, żeby biblioteka miała gdzie wypisywać informacje
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

In [26]:
train, test = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
# przygotowanie słowników interakcji, aby wiedzieć, co użytkownik już widział
# train_ur: user -> [items in train]
train_ur = {}

for u, i in zip(train['user'], train['item']):
    train_ur[u] = set()
    train_ur[u].add(i)

test_ur = {}

for u, i in zip(test['user'], test['item']):
    test_ur[u] = []
    test_ur[u].append(i)

test_u_list = list(test_ur.keys())

In [ ]:
config = {
    'algo_name': 'itemknn',
    'test_method': 'rsbr',
    'test_size': 0.2,
    'topk': -1,
    'maxk': 20,
    'sim_method': 'cosine',
    'user_num': num_users,
    'item_num': num_items,
    'gpu': '0',       # ID karty graficznej
    'logger': logger, 
    'shrink': 0,            
    'similarity': 'cosine', 
    'normalize': True,     
}

In [16]:
# itemknn
iknn = ItemKNNCF(config=config)
iknn.fit(train)
all_scores = iknn.pred_mat.toarray()

INFO:root:ItemKNNCFRecommender: Detected 23 (0.62 %) cold items.
INFO:root:Similarity column 3706 ( 100 % ), 894.23 column/sec, elapsed time 0.07 min


In [ ]:
config = {
    'algo_name': 'userknn',  
    'topk': 4000, #wcześniej -1
    'maxk': 20,              # liczba sąsiadów
    'gpu': '0',
    'logger': logger,
    'user_num': num_users,
    'item_num': num_items,
    'shrink': 0,
    'similarity': 'cosine',  # możesz być 'pearson'
    'normalize': True,
}

In [30]:
# uknn
uknn = UserKNNCF(config=config)
uknn.fit(train)
all_scores = uknn.pred_mat.toarray()

INFO:root:UserKNNCFRecommender: Detected 23 (0.62 %) cold users.
INFO:root:Similarity column 6040 ( 100 % ), 807.85 column/sec, elapsed time 0.12 min


In [53]:
config = {
    'algo_name': 'puresvd',
    'topk': 10,
    'gpu': '0',
    'logger': logger,
    'user_num': num_users,
    'item_num': num_items,
    'factors': 50,    # l. ukrytych wymiarów 
}

In [58]:
svd = PureSVD(config=config)
svd.fit(train)

INFO:root:Computing SVD decomposition...
INFO:root:Finish build train matrix for decomposition
INFO:root:Done!


In [ ]:
# pobranie wektorów z modelu
user_emb = svd.user_vec
item_emb = svd.item_vec

# mnożenie macierzy, wzór: Score = UserVector * ItemVector_Transposed
# (liczbaU x cechy) * (cechy x liczbaFilmów) = (liczbaU x liczbaFilmów)
all_scores = np.dot(user_emb, item_emb.T)

In [ ]:
#  maskowanie: Ustawiamy -inf dla filmów, które były w treningu 
for u, items in train_ur.items():
    all_scores[u, list(items)] = -np.inf

test_scores = all_scores[test_u_list]

# argsort sortuje rosnąco, więc bierzemy ostatnie K elementów i odwracamy kolejność (::-1)
pred_ur_matrix = np.argsort(test_scores, axis=1)[:, -config['topk']:][:, ::-1]

In [ ]:
y_true = []
y_pred = []

for row in test[['user', 'item', 'rating']].values:
    u = int(row[0])  # user ID
    i = int(row[1])  # item ID
    r = float(row[2]) # prawdziwa ocena

    pred = all_scores[u, i]

    if np.isnan(pred):
        pred = 0.0 # lub średnia globalna

    y_true.append(r)
    y_pred.append(pred)

# obliczenie metryk
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)

print(f"--- WYNIKI ---")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")

--- WYNIKI ---
RMSE: 21.6463
MAE:  9.5129


In [13]:
np.asfarray = np.asarray
precision_score = Precision(test_ur, pred_ur_matrix, test_u_list)
recall_score = Recall(test_ur, pred_ur_matrix, test_u_list)
ndcg_score = NDCG(test_ur, pred_ur_matrix, test_u_list)

print(f"Precision@{config['topk']}: {precision_score:.4f}")
print(f"Recall@{config['topk']}:    {recall_score:.4f}")
print(f"NDCG@{config['topk']}:      {ndcg_score:.4f}")

Precision@10: 0.3235
Recall@10:    0.1416
NDCG@10:      0.6364


In [ ]:
config = {
    'algo_name': 'mostpop',
    'topk': 5,
    'gpu': '0',
    'logger': logger,
    'user_num': num_users,
    'item_num': num_items,
    'UID_NAME': 'user',    # nazwa kolumny użytkownika
    'IID_NAME': 'item',    # nazwa kolumny przedmiotu 
    'INTER_NAME': 'rating'
}

In [21]:
mostPop = MostPop(config=config)
mostPop.fit(train)

In [ ]:
pop_scores = mostPop.item_score
num_test_users = len(test_u_list)
all_scores = np.tile(pop_scores, (num_test_users, 1))
for idx, u in enumerate(test_u_list):
    items_to_mask = list(train_ur[u])
    all_scores[idx, items_to_mask] = -np.inf

pred_ur_matrix = np.argsort(all_scores, axis=1)[:, -config['topk']:][:, ::-1]

In [23]:
np.asfarray = np.asarray

In [24]:
precision_score = Precision(test_ur, pred_ur_matrix, test_u_list)
recall_score = Recall(test_ur, pred_ur_matrix, test_u_list)
ndcg_score = NDCG(test_ur, pred_ur_matrix, test_u_list)

print(f"Precision@{config['topk']}: {precision_score:.4f}")
print(f"Recall@{config['topk']}:    {recall_score:.4f}")
print(f"NDCG@{config['topk']}:      {ndcg_score:.4f}")

Precision@5: 0.2100
Recall@5:    0.0402
NDCG@5:      0.4005
